Montgomery County Maryland Wine Sales Dashboard V2

Module imports

In [9]:
import pandas as pd
import numpy as np
import requests
import PyPDF2
import io
import re
import sqlite3
import time
import datetime
import plotly.graph_objects as go
import plotly.express as px
import warnings
from plotly.subplots import make_subplots
from difflib import SequenceMatcher
from datetime import datetime
from openpyxl import Workbook
from io import StringIO
warnings.filterwarnings('ignore')

The below functions are standard for all tables, just a bit of code to help with cleaning and validation.

File imports

In [15]:
#GitHub raw URL (not the web interface URL)
base_url = "https://raw.githubusercontent.com/ac604605/Montgomery_County_Dashboard/main/"

# Load Files Directly
try:
    Distributors_Virginia_Three_Main = pd.read_csv(base_url + "data/Distributors_Virginia_Three_Main.csv")
    print(f"Distributors loaded: {Distributors_Virginia_Three_Main.shape}")
    
    wine_producers = pd.read_csv(base_url + "wine_producers.csv")
    print(f"Wine Producers loaded: {wine_producers.shape}")

    Warehouse_and_Retail_Sales = pd.read_csv(base_url + "data/Warehouse_and_Retail_Sales.csv")
    print(f"Warehouse and Retail Sales loaded: {Warehouse_and_Retail_Sales.shape}")

    Wine_Review_Data = pd.read_csv(base_url + "data/winemag-data-130k-v2.csv/winemag-data-130k-v2.csv")
    print(f"Wine Review Data loaded: {Wine_Review_Data.shape}")

    Wine_Review_Data = pd.read_csv(base_url + "data/winemag-data-130k-v2.csv/winemag-data-130k-v2.csv")
    print(f"Wine Review Data loaded: {Wine_Review_Data.shape}")

     # Fix mixed types warning
    Suppliers_Importers_Retailers = pd.read_csv(base_url + "data/Suppliers_Importers_Retailers.csv", 
                                               low_memory=False)
    print(f"Suppliers/Importers/Retailers loaded: {Suppliers_Importers_Retailers.shape}")
    
except Exception as e:
    print(f"Error loading data: {e}")
    print("Check that the file exists in your GitHub repository")

# Display first few rows to verify
print("\nFirst 5 rows of Distributors data:")
print(Distributors_Virginia_Three_Main.head())
print("\nFirst 5 rows of Wine Producer data:")
print(wine_producers.head())
print("\nFirst 5 rows of Montgomery  data:")
print(Warehouse_and_Retail_Sales.head())
print("\nFirst 5 rows of Warehouse and Retail Sales data:")
print(Wine_Review_Data.head())
print("\nFirst 5 rows of Supplier data:")
print(Suppliers_Importers_Retailers.head())

Distributors loaded: (14279, 4)
Wine Producers loaded: (2324, 7)
Warehouse and Retail Sales loaded: (307645, 9)
Wine Review Data loaded: (129971, 14)
Suppliers/Importers/Retailers loaded: (78067, 7)

First 5 rows of Distributors data:
   License_ID             Brand_Name Product_Type Distributor
0       85629              #NICEWINE         Wine        RNDC
1       85629                 10SPAN         Wine        RNDC
2       85629         12 GENERATIONS         Wine        RNDC
3       85629             12 KNIGHTS         Wine        RNDC
4       85629  12 WINES OF CHRISTMAS         Wine        RNDC

First 5 rows of Wine Producer data:
  License ID               Trade Name                   Address        City  \
0     X14533         1026 beverage co           839 S Beacon St   San Pedro   
1     X11997                  117 LLC  1352 Main Street Suite 1  Rutherford   
2     X15111        1413 VALA  AVENUE                       NaN        Napa   
3     X14269  2 X 4 BREWING & IMPORTS   

Validate all loaded datasets.

In [19]:
print("\nVALIDATING ALL DATASETS...")
print("=" * 60)

# Create a list of all your datasets with their names
datasets = [
    ('Distributors_Virginia_Three_Main', Distributors_Virginia_Three_Main),
    ('wine_producers', wine_producers),
    ('Warehouse_and_Retail_Sales', Warehouse_and_Retail_Sales),
    ('Wine_Review_Data', Wine_Review_Data),
    ('Suppliers_Importers_Retailers', Suppliers_Importers_Retailers)
    # Add more datasets here as needed
]

# Quick validation of all datasets
for name, df in datasets:
    print(f"\n{name.upper()}:")
    print("-" * 40)
    quick_validate(df)
    print()

print("Initial validation complete! Use validate_dataframe(df) for detailed analysis of any dataset.")


VALIDATING ALL DATASETS...

DISTRIBUTORS_VIRGINIA_THREE_MAIN:
----------------------------------------
Shape: (14279, 4)
Missing values: 0
Duplicates: 5132
Data types: {dtype('O'): 3, dtype('int64'): 1}
Numeric ranges:
     License_ID
min     85628.0
max  13258386.0


WINE_PRODUCERS:
----------------------------------------
Shape: (2324, 7)
Missing values: 4
Duplicates: 0
Data types: {dtype('O'): 7}


WAREHOUSE_AND_RETAIL_SALES:
----------------------------------------
Shape: (307645, 9)
Missing values: 171
Duplicates: 0
Data types: {dtype('O'): 4, dtype('float64'): 3, dtype('int64'): 2}
Numeric ranges:
       YEAR  MONTH  RETAIL SALES  RETAIL TRANSFERS  WAREHOUSE SALES
min  2017.0    1.0         -6.49            -38.49          -7800.0
max  2020.0   12.0       2739.00           1990.83          18317.0


WINE_REVIEW_DATA:
----------------------------------------
Shape: (129971, 14)
Missing values: 204752
Duplicates: 0
Data types: {dtype('O'): 11, dtype('int64'): 2, dtype('float64'): 

Run standardized data cleaning processes.

In [20]:
df_cleaned, report = run_phase_1_2_cleaning(
    df=Warehouse_and_Retail_Sales,
    table_name="Warehouse_and_Retail_Sales"
)


############################################################
DATA CLEANING PHASES 1 & 2: Warehouse_and_Retail_Sales
############################################################

PHASE 1: INITIAL ASSESSMENT - Warehouse_and_Retail_Sales
Dataset Shape: (307645, 9)
Memory Usage: 87.55 MB

Data Types Distribution:
  object: 4 columns
  float64: 3 columns
  int64: 2 columns

Missing Values by Column:
  SUPPLIER: 167 (0.05%)
  RETAIL SALES: 3 (0.00%)
  ITEM TYPE: 1 (0.00%)

Duplicate Rows: 0

Column Details:
  YEAR: int64 | 4 unique values
  MONTH: int64 | 12 unique values
  SUPPLIER: object | 396 unique values
  ITEM CODE: object | 34056 unique values
  ITEM DESCRIPTION: object | 34822 unique values
  ITEM TYPE: object | 8 unique values
  RETAIL SALES: float64 | 10674 unique values
  RETAIL TRANSFERS: float64 | 2504 unique values
  WAREHOUSE SALES: float64 | 4895 unique values

DATA DICTIONARY TEMPLATE - Warehouse_and_Retail_Sales
          column data_type    description business_rules    

Begin basic manual exploration to determine missing values. Uncomment various lines to review data. 

In [49]:
df=Warehouse_and_Retail_Sales

#df.head()
#df.info()
#df.describe()
#df.isnull().sum()
#df['ITEM TYPE'].value_counts()
#quick_item_type_examples(df, n_examples=5)
#basic_data_exploration(df)
manual_column_investigation(df)


MANUAL COLUMN-BY-COLUMN INVESTIGATION

--- YEAR ---
Type: int64
Non-null count: 307645
Unique values: 4
Value counts:
YEAR
2019    138638
2017     96284
2020     46278
2018     26445
Name: count, dtype: int64
Range: 2017 to 2020
------------------------------

--- MONTH ---
Type: int64
Non-null count: 307645
Unique values: 12
Value counts:
MONTH
1     37724
9     37380
7     36710
11    27381
10    26712
8     26003
6     25958
2     25389
3     24206
12    14500
Name: count, dtype: int64
Range: 1 to 12
------------------------------

--- SUPPLIER ---
Type: object
Non-null count: 307478
Unique values: 396
Value counts:
SUPPLIER
REPUBLIC NATIONAL DISTRIBUTING CO       20995
LEGENDS LTD                             13634
SOUTHERN GLAZERS WINE AND SPIRITS       11720
E & J GALLO WINERY                      10816
THE COUNTRY VINTNER, LLC DBA WINEBOW    10669
MONSIEUR TOUTON SELECTION               10360
A VINTNERS SELECTIONS                    9994
BACCHUS IMPORTERS LTD                    

In [45]:
def basic_data_exploration(df):
    """
    Generic exploration that works on any dataset
    """
    print("="*50)
    print("BASIC DATA EXPLORATION")
    print("="*50)
    
    print("Dataset shape:", df.shape)
    print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
    
    print("\nFirst few rows:")
    print(df.head())
    
    print("\nData types and non-null counts:")
    print(df.info())
    
    print("\nBasic statistics:")
    print(df.describe(include='all'))
    
    print("\nMissing values:")
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) > 0:
        print(missing)
    else:
        print("No missing values")
    
    print("\nDuplicate rows:", df.duplicated().sum())

def manual_column_investigation(df):
    """
    Go through each column manually - this is what analysts actually do
    """
    print("\n" + "="*50)
    print("MANUAL COLUMN-BY-COLUMN INVESTIGATION")
    print("="*50)
    
    for col in df.columns:
        print(f"\n--- {col} ---")
        print(f"Type: {df[col].dtype}")
        print(f"Non-null count: {df[col].count()}")
        print(f"Unique values: {df[col].nunique()}")
        
        # Show value counts for categorical-like columns
        if df[col].dtype == 'object' or df[col].nunique() < 20:
            print("Value counts:")
            print(df[col].value_counts().head(10))
        else:
            print("Sample values:")
            print(df[col].dropna().head(5).tolist())
        
        # Check for obvious issues
        if df[col].dtype in ['int64', 'float64']:
            print(f"Range: {df[col].min()} to {df[col].max()}")
            if df[col].min() < 0:
                print("⚠️  Contains negative values")
        
        print("-" * 30)

def investigate_missing_values_manually(df):
    """
    Manual investigation of missing values - the detective work
    """
    print("\n" + "="*50)
    print("MANUAL MISSING VALUES INVESTIGATION")
    print("="*50)
    
    # Find columns with missing values
    missing_cols = df.columns[df.isnull().any()].tolist()
    
    if not missing_cols:
        print("No missing values to investigate")
        return
    
    print(f"Columns with missing values: {missing_cols}")
    
    for col in missing_cols:
        print(f"\n🔍 Investigating {col}:")
        print(f"Missing count: {df[col].isnull().sum()}")
        
        # Look at rows with missing values
        missing_rows = df[df[col].isnull()]
        print(f"Sample rows with missing {col}:")
        
        # Show other columns for context
        other_cols = [c for c in df.columns if c != col][:5]  # Show first 5 other columns
        print(missing_rows[other_cols].head(3))
        
        # Look for patterns
        print(f"\nLooking for patterns in missing {col}:")
        
        # Check if other columns have values when this one is missing
        for other_col in other_cols:
            if df[other_col].dtype == 'object' or df[other_col].nunique() < 50:
                pattern = missing_rows[other_col].value_counts().head(3)
                if len(pattern) > 0:
                    print(f"  When {col} is missing, {other_col} is often: {pattern.index[0]}")

def spot_anomalies_manually(df):
    """
    Manual anomaly detection - what analysts look for
    """
    print("\n" + "="*50)
    print("MANUAL ANOMALY DETECTION")
    print("="*50)
    
    # Check for suspicious patterns
    print("🔍 Looking for suspicious patterns:")
    
    # Numeric columns with zeros
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        zero_count = (df[col] == 0).sum()
        if zero_count > 0:
            zero_pct = zero_count / len(df) * 100
            print(f"  {col}: {zero_count} zeros ({zero_pct:.1f}%)")
    
    # Text columns with unusual patterns
    text_cols = df.select_dtypes(include=['object']).columns
    for col in text_cols:
        print(f"\n🔍 Examining {col} for patterns:")
        
        # Look at length distribution
        lengths = df[col].str.len()
        print(f"  Text length range: {lengths.min()} to {lengths.max()}")
        
        # Look for common words/patterns
        if col in ['ITEM DESCRIPTION', 'DESCRIPTION']:  # Common column names
            # Sample some values manually
            sample_values = df[col].dropna().head(10).tolist()
            print(f"  Sample values: {sample_values[:3]}")
            
            # Look for obvious non-alcoholic items
            non_alcoholic_keywords = ['opener', 'glass', 'mixer', 'tool', 'accessory']
            for keyword in non_alcoholic_keywords:
                matches = df[col].str.contains(keyword, case=False, na=False).sum()
                if matches > 0:
                    print(f"  ⚠️  Found {matches} items containing '{keyword}'")

def ask_business_questions(df):
    """
    The questions analysts ask themselves
    """
    print("\n" + "="*50)
    print("BUSINESS QUESTIONS TO INVESTIGATE")
    print("="*50)
    
    print("Questions I would ask about this dataset:")
    print("1. What time period does this cover?")
    print("2. Are all these items actually alcoholic beverages?")
    print("3. Why are some suppliers missing - are these different types of items?")
    print("4. Are zero sales values real or missing data?")
    print("5. Do item codes follow a pattern?")
    print("6. Are there seasonal patterns in the data?")
    
    # Try to answer some automatically
    print("\nQuick answers from the data:")
    
    # Time period
    if 'YEAR' in df.columns:
        years = df['YEAR'].unique()
        print(f"  Time period: {min(years)} to {max(years)}")
    
    # Item types
    if 'ITEM TYPE' in df.columns:
        item_types = df['ITEM TYPE'].value_counts()
        print(f"  Item types: {list(item_types.index)}")
    
    # Suppliers
    if 'SUPPLIER' in df.columns:
        supplier_count = df['SUPPLIER'].nunique()
        print(f"  Number of suppliers: {supplier_count}")

# ================================
# NOTEBOOK-STYLE EXPLORATION
# ================================

def explore_like_a_real_analyst(df):
    """
    This is how a real analyst would explore unknown data
    """
    print("REAL-WORLD DATA EXPLORATION PROCESS")
    print("="*60)
    
    # Step 1: Always start with basics
    basic_data_exploration(df)
    
    # Step 2: Look at each column manually
    manual_column_investigation(df)
    
    # Step 3: Investigate missing values
    investigate_missing_values_manually(df)
    
    # Step 4: Look for anomalies
    spot_anomalies_manually(df)
    
    # Step 5: Ask business questions
    ask_business_questions(df)
    
    print("\n" + "="*60)
    print("NEXT STEPS:")
    print("="*60)
    print("Based on this exploration, I would:")
    print("1. Deep-dive into the missing supplier pattern")
    print("2. Investigate item types that aren't beer/wine")
    print("3. Look at sales patterns across different categories")
    print("4. Create targeted cleaning functions")
    print("5. Document business rules for filtering")

# ================================
# USAGE
# ================================

# This is how you'd actually start with unknown data:
#explore_like_a_real_analyst(Warehouse_and_Retail_Sales)

# Then based on what you find, you'd write specific investigation code
# like what I provided earlier

In [58]:
import os
print(os.getcwd())

C:\Users\LOW14


In [56]:
import data_exploration_utils.py

ModuleNotFoundError: No module named 'data_exploration_utils'

In [50]:
import data_exploration_utils as deu

ModuleNotFoundError: No module named 'data_exploration_utils'